In [2]:
#!pip install --upgrade gspread pandas gspread_dataframe
import numpy as np
import pandas as pd

import gspread
from google.auth import default
from google.colab import auth

# Google Drive & Google Sheets API 인증
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Google Sheets 파일 열기
spreadsheet_id = "12RmvY2KI-6UcMEHXX4E65R_oso4965jNpgAbKeIgI9c"  # Google Sheets의 ID
spreadsheet = gc.open_by_key(spreadsheet_id)  # 파일 열기

# 특정 시트 선택 및 데이터 가져오기
worksheet = spreadsheet.sheet1  # 첫번째 시트 선택
data = worksheet.get_all_values()  # 데이터 가져오기

# DataFrame 형식으로 변환하고, 첫번째 행을 이 데이터의 헤더로 지정
df = pd.DataFrame(data[1:], columns=data[0])

# 데이터 타입 변환
df['BMI'] = df['BMI'].astype(float)
df['Exercise_per_week'] = df['Exercise_per_week'].astype(int)
df['Hypertension_risk'] = df['Hypertension_risk'].astype(float)

# 범주형 데이터 변환 (Label Encoding)
df['Diet'] = df['Diet'].map({'Poor': 0, 'Average': 1, 'Good': 2})  # 'Diet' 값을 숫자로 변환

print("\n<읽어온 데이터 확인>")
print(df)

# 특성값 데이터(X)와 레이블 데이터(y) 설정
X = df[['BMI', 'Exercise_per_week', 'Diet']].values
y = df['Hypertension_risk'].values

# 가중치와 bias 초기화 (weight는 각 feature에 대한 파라미터, bias는 절편)
n_features = X.shape[1]
weight = np.random.rand(n_features)
bias = np.random.rand()

# 학습 진행을 위한 조건 설정
learning_rate = 0.001  # 학습률
epochs = 2000  # 학습횟수
m = X.shape[0]  # 훈련 데이터 수

print("\n<학습 진행 중 weight, bias, cost의 변화 과정 보기>")

# 경사하강법 학습 과정
for epoch in range(epochs):
    # 예측값 계산: 각 샘플에 대해 np.dot(X, weight) + bias
    y_pred = np.dot(X, weight) + bias

    # 비용함수 계산 (평균제곱오차, MSE)
    cost = np.mean((y_pred - y) ** 2)

    # 기울기 계산 (각 파라미터별로)
    weight_grad = (2/m) * np.dot(X.T, (y_pred - y))
    bias_grad = (2/m) * np.sum(y_pred - y)

    # 파라미터 업데이트
    weight -= learning_rate * weight_grad
    bias   -= learning_rate * bias_grad

    # 100 에폭마다 진행 상황 출력
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:004d}/{epochs}, Weights: {[f'{val:.4f}' for val in weight]}, Bias: {bias:.4f}, Cost: {cost:.4f}")

print("\n<학습을 통해 찾아진 파라미터 최종값>")
print(f"Final Weights: {[f'{val:.4f}' for val in weight]}")
print(f"Final Bias: {bias:.4f}")

# 모델 테스트
print("\n▶ 모델 테스트")
test_input = np.array([28.5, 3, 1]) # 테스트 데이터 입력
print(f"'BMI: {test_input[0]}, 주간운동시간: {test_input[1]}시간, 식습관: {test_input[2]}'인 환자의 고혈압 발병 확률은?")
predicted_risk = np.dot(test_input, weight) + bias
print(f" → {predicted_risk*100:.2f}%입니다.")


<읽어온 데이터 확인>
    BMI  Exercise_per_week  Diet  Hypertension_risk
0  22.5                  3     1               0.25
1  28.0                  1     0               0.75
2  26.0                  2     1               0.60
3  30.5                  0     0               0.90
4  24.0                  4     2               0.20
5  27.5                  1     0               0.80
6  25.5                  3     1               0.45
7  23.0                  5     2               0.15
8  29.0                  0     0               0.85
9  21.5                  6     2               0.10

<학습 진행 중 weight, bias, cost의 변화 과정 보기>
Epoch 0100/2000, Weights: ['0.0168', '-0.1165', '0.3769'], Bias: 0.0556, Cost: 0.1204
Epoch 0200/2000, Weights: ['0.0254', '-0.1983', '0.3379'], Bias: 0.0519, Cost: 0.0271
Epoch 0300/2000, Weights: ['0.0282', '-0.2233', '0.3219'], Bias: 0.0507, Cost: 0.0172
Epoch 0400/2000, Weights: ['0.0291', '-0.2300', '0.3133'], Bias: 0.0503, Cost: 0.0159
Epoch 0500/2000, Weights: ['0.